## Show each customers running wallet balance after each transaction

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

# Sample customer transactions data
data = [
    (1, "2026-07-01", "credit", 100),
    (1, "2026-07-02", "debit", 30),
    (1, "2026-07-03", "credit", 50),
    (1, "2026-07-04", "debit", 20),
    (2, "2026-07-01", "credit", 200),
    (2, "2026-07-02", "debit", 80),
    (2, "2026-07-03", "credit", 40),
    (2, "2026-07-04", "debit", 60)
]

columns = ["customer_id", "transaction_date", "transaction_type", "amount"]
df = spark.createDataFrame(data, columns)

# Calculate credit and debit columns
df = df.withColumn("credit", when(col("transaction_type") == "credit", col("amount")).otherwise(0)) \
       .withColumn("debit", when(col("transaction_type") == "debit", col("amount")).otherwise(0))

# Calculate running wallet balance
window_spec = (
    Window.partitionBy("customer_id")
    .orderBy("transaction_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df = df.withColumn(
    "running_balance",
    spark_sum(col("credit") - col("debit")).over(window_spec)
)

display(df)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

# Sample customer transactions data
data = [
    (1, "2026-07-01", "credit", 100),
    (1, "2026-07-02", "debit", 30),
    (1, "2026-07-03", "credit", 50),
    (1, "2026-07-04", "debit", 20),
    (2, "2026-07-01", "credit", 200),
    (2, "2026-07-02", "debit", 80),
    (2, "2026-07-03", "credit", 40),
    (2, "2026-07-04", "debit", 60)
]

columns = ["customer_id", "transaction_date", "transaction_type", "amount"]
df = spark.createDataFrame(data, columns)
df.createOrReplaceTempView("transactions")
display(df)
# DBTITLE 1,Write a SQL query to calculate the total credit and debit amounts for each customer in the last 30 days.
# Calculate credit and debit columns

# DBTITLE 1,Write a SQL query to calculate the total credit and debit amounts for each customer in the last 30 days.

In [0]:
%sql
select *
from transactions order by customer_id, transaction_date


In [0]:
%sql
select *,
sum(amount) over(partition by customer_id order by transaction_date) as running_balance
from transactions order by customer_id, transaction_date asc

In [0]:
%sql
select *,
sum(
    case 
        when transaction_type = 'credit' then amount
        when transaction_type = 'debit' then -amount 
        else 0
    end
) over(partition by customer_id order by transaction_date) as running_balance
from transactions
order by customer_id, transaction_date asc

## Find all numbers that appears at least 3 times in a row

In [0]:
data = [
    (5,),
    (5,),
    (5,),
    (8,),
    (8,),
    (6,),
    (12,),
    (12,),
    (12,),
    (12,),
    (6,),
    (1,),
    (1,),
    (1,)
]
columns = ["num"]
df = spark.createDataFrame(data, columns)
df.createOrReplaceTempView('numb')
display(df)

In [0]:
%sql

select num,
lead(num,1) over(order by num) as next1,
lead(num,2) over(order by num) as next2
from numb

In [0]:
%sql

with tbl as (select num,
lead(num,1) over(order by num) as next1,
lead(num,2) over(order by num) as next2
from numb)

select distinct num from tbl
where num =next1 and num=next2 and next1=next2

### Find the first and last orders placed by each customer

In [0]:
from datetime import datetime, timedelta

# Sample product order data with customers having frequent orders
order_data = [
    (1, "2026-07-01", "A", 2),
    (1, "2026-07-02", "B", 1),
    (1, "2026-07-03", "A", 3),
    (1, "2026-07-04", "C", 1),
    (1, "2026-07-05", "B", 2),
    (2, "2026-07-01", "A", 1),
    (2, "2026-07-02", "A", 2),
    (2, "2026-07-03", "B", 1),
    (2, "2026-07-04", "C", 2),
    (2, "2026-07-05", "C", 1),
    (3, "2026-07-01", "D", 1),
    (3, "2026-07-03", "D", 2),
    (3, "2026-07-05", "E", 1),
    (4, "2026-07-01", "F", 1)
]

order_columns = ["customer_id", "order_date", "product_id", "quantity"]
order_df = spark.createDataFrame(order_data, order_columns)
order_df.createOrReplaceTempView("product_orders")
display(order_df)

In [0]:
from pyspark.sql.functions import min, max

# Find first and last order date for each customer using PySpark
result_df = order_df.groupBy("customer_id").agg(
    min("order_date").alias("first_order"),
    max("order_date").alias("last_order")
)

display(result_df)

In [0]:
%sql
select customer_id , order_date from product_orders group by customer_id, order_date order by customer_id, order_date asc

In [0]:
%sql
select customer_id , order_date, row_number() over(partition by customer_id order by order_date asc) as rnk from product_orders group by customer_id, order_date order by customer_id, order_date asc 

In [0]:
%sql
with tbl as (
select customer_id , order_date, row_number() over(partition by customer_id order by order_date asc) as rnk from product_orders group by customer_id, order_date order by customer_id, order_date asc )

select customer_id,
min(order_date) as first_order,
max(order_date) as last_order
 from tbl group by customer_id

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
order_df=order_df.groupBy("customer_id").agg(min(col('order_date')).alias("first_order"),max(col('order_date')).alias("last_order"))
display(order_df.show(truncate=False))

### Calculate the total trips, cancelled trips, and trip cancellation rate for each city 

In [0]:
import random

trip_data = [
    (1, "2026-07-01", "New York", 101, 201, "completed"),
    (2, "2026-07-01", "New York", 102, 202, "cancelled"),
    (11, "2026-07-02", "New York", 103, 203, "Completed"),
    (12, "2026-07-03", "New York", 104, 204, "CANCELLED"),
    (3, "2026-07-02", "Los Angeles", 105, 205, "complete"),
    (4, "2026-07-02", "Los Angeles", 106, 206, "Cancel"),
    (13, "2026-07-03", "Los Angeles", 107, 207, "COMPLETED"),
    (14, "2026-07-04", "Los Angeles", 108, 208, "cancelled"),
    (5, "2026-07-03", "Chicago", 109, 209, "Completed"),
    (6, "2026-07-03", "Chicago", 110, 210, "Cancelled"),
    (15, "2026-07-04", "Chicago", 111, 211, "completed"),
    (16, "2026-07-05", "Chicago", 112, 212, "CANCELLED"),
    (7, "2026-07-04", "Houston", 113, 213, "complete"),
    (8, "2026-07-04", "Houston", 114, 214, "Cancel"),
    (17, "2026-07-05", "Houston", 115, 215, "COMPLETED"),
    (18, "2026-07-06", "Houston", 116, 216, "cancelled"),
    (9, "2026-07-05", "Phoenix", 117, 217, "Completed"),
    (10, "2026-07-05", "Phoenix", 118, 218, "Cancelled"),
    (19, "2026-07-06", "Phoenix", 119, 219, "completed"),
    (20, "2026-07-07", "Phoenix", 120, 220, "CANCELLED")
]

trip_columns = ["trip_id", "trip_date", "city", "rider_id", "driver_id", "trip_status"]
trip_df = spark.createDataFrame(trip_data, trip_columns)
display(trip_df)

trip_df.createOrReplaceTempView("trips")

In [0]:
%sql 
with tbl as (select city, count(trip_id) as total_trip_cnt, sum(case when lower(trip_status)='cancelled' then 1 else 0 end ) as cancelled_trip_cnt from trips group by city order by count(trip_id))

select city, total_trip_cnt,cancelled_trip_cnt, cancelled_trip_cnt*100/total_trip_cnt as rate_of_cancellation from tbl



### Find the average delivery time (in minutes) per restaurant.

In [0]:
from pyspark.sql.functions import col, avg

# Sample food delivery data
delivery_data = [
    (1, "2026-07-01 12:00:00", "2026-07-01 12:30:00", "Pizza Palace"),
    (2, "2026-07-01 13:00:00", "2026-07-01 13:40:00", "Pizza Palace"),
    (3, "2026-07-01 14:00:00", "2026-07-01 14:20:00", "Burger Barn"),
    (4, "2026-07-01 15:00:00", "2026-07-01 15:35:00", "Burger Barn"),
    (5, "2026-07-01 16:00:00", "2026-07-01 16:25:00", "Sushi Spot"),
    (6, "2026-07-01 17:00:00", "2026-07-01 17:50:00", "Sushi Spot"),
    (7, "2026-07-02 12:00:00", "2026-07-02 12:35:00", "Pizza Palace"),
    (8, "2026-07-02 13:00:00", "2026-07-02 13:45:00", "Burger Barn"),
    (9, "2026-07-02 14:00:00", "2026-07-02 14:30:00", "Pizza Palace"),
    (10, "2026-07-02 15:00:00", "2026-07-02 15:40:00", "Sushi Spot"),
    (11, "2026-07-02 16:00:00", "2026-07-02 16:50:00", "Burger Barn"),
    (12, "2026-07-02 17:00:00", "2026-07-02 17:30:00", "Pizza Palace"),
    (13, "2026-07-03 12:00:00", "2026-07-03 12:25:00", "Sushi Spot"),
    (14, "2026-07-03 13:00:00", "2026-07-03 13:35:00", "Sushi Spot"),
    (15, "2026-07-03 14:00:00", "2026-07-03 14:40:00", "Pizza Palace"),
    (16, "2026-07-03 15:00:00", "2026-07-03 15:30:00", "Pizza Palace")
]

delivery_columns = ["order_id", "order_time", "delivery_time", "restaurant"]
delivery_df = spark.createDataFrame(delivery_data, delivery_columns)
delivery_df.createOrReplaceTempView("food_deliveries")
display(delivery_df)

In [0]:
%sql
with tbl as (
select restaurant,order_id,order_time,delivery_time,timestampdiff(minute,order_time,delivery_time) as time_taken,count(*) over(partition by restaurant) as order_cnt from food_deliveries
)

select * from tbl 


In [0]:
%sql

with tbl as (
select restaurant,order_id,order_time,delivery_time,timestampdiff(minute,order_time,delivery_time) as time_taken,count(*) over(partition by restaurant) as order_cnt from food_deliveries
)

select restaurant, sum(time_taken)/order_cnt as avg_time from tbl group by restaurant,order_cnt

## find total revenue by city, total revnue by customer, by product, by month

In [0]:
# Sample customers dataset
customers_data = [
    (1, "Alice", "Smith", "Acme Corp", "New York", "USA", "alice.smith@acme.com", "acme.com"),
    (2, "Bob", "Johnson", "Beta LLC", "Los Angeles", "USA", "bob.johnson@beta.com", "beta.com"),
    (3, "Carol", "Williams", "Gamma Inc", "Chicago", "USA", "carol.williams@gamma.com", "gamma.com"),
    (4, "David", "Brown", "Delta Ltd", "Houston", "USA", "david.brown@delta.com", "delta.com"),
    (5, "Eve", "Davis", "Epsilon PLC", "Phoenix", "USA", "eve.davis@epsilon.com", "epsilon.com"),
    (6, "Frank", "Miller", "Zeta AG", "San Francisco", "USA", "frank.miller@zeta.com", "zeta.com"),
    (7, "Grace", "Wilson", "Eta GmbH", "Seattle", "USA", "grace.wilson@eta.com", "eta.com"),
    (8, "Hank", "Moore", "Theta BV", "Boston", "USA", "hank.moore@theta.com", "theta.com"),
    (9, "Ivy", "Taylor", "Iota SAS", "Miami", "USA", "ivy.taylor@iota.com", "iota.com"),
    (10, "Jack", "Anderson", "Kappa Ltd", "Denver", "USA", "jack.anderson@kappa.com", "kappa.com")
]
customers_columns = ["Customer_Id", "First_Name", "Last_Name", "Company", "City", "Country", "Email", "Website"]
customers_df = spark.createDataFrame(customers_data, customers_columns)
customers_df.createOrReplaceTempView("customers")
display(customers_df)

# Sample products dataset
products_data = [
    (101, "Laptop", "Electronics", 1200.00),
    (102, "Smartphone", "Electronics", 800.00),
    (103, "Desk Chair", "Furniture", 150.00)
]
products_columns = ["Product_Id", "Product_Name", "Category", "Price"]
products_df = spark.createDataFrame(products_data, products_columns)
products_df.createOrReplaceTempView("products")
display(products_df)

# Sample orders dataset with more rows for different months
orders_data = [
    (1001, 1, 101, "2026-07-01", 2),
    (1002, 2, 102, "2026-07-02", 1),
    (1003, 9, 103, "2026-07-03", 4),
    (1004, 5, 103, "2026-07-04", 1),
    (1005, 2, 101, "2026-08-05", 1),
    (1006, 6, 102, "2026-08-10", 2),
    (1007, 4, 102, "2026-09-15", 3),
    (1008, 2, 103, "2026-09-20", 2),
    (1009, 7, 101, "2026-10-01", 1),
    (1010, 1, 101, "2026-10-10", 2),
    (1011, 8, 102, "2026-11-05", 1),
    (1012, 3, 103, "2026-11-15", 3),
    (1013, 10, 103, "2026-12-01", 2),
    (1014, 2, 101, "2026-12-10", 1),
    (1015, 3, 102, "2026-12-20", 2)
]
orders_columns = ["Order_Id", "Customer_Id", "Product_Id", "Order_Date", "Quantity"]
orders_df = spark.createDataFrame(orders_data, orders_columns)
orders_df.createOrReplaceTempView("orders")
display(orders_df)

In [0]:
%sql

select  o.*,product_name,price,category,First_Name,Last_Name,Company,City,Country from orders o
left join(select product_id, product_name,price,category from products) p on o.product_id=p.product_id
left join (select Customer_Id,First_Name,Last_Name,Company,City,Country from customers) c on o.customer_id=c.customer_id





In [0]:
%sql

with final_tbl as (
    select  o.*,product_name,price,category,First_Name,Last_Name,Company,City,Country from orders o
left join(select product_id, product_name,price,category from products) p on o.product_id=p.product_id
left join (select Customer_Id,First_Name,Last_Name,Company,City,Country from customers) c on o.customer_id=c.customer_id
)

select order_id,product_id,product_name,city,price,quantity, quantity*price as total_revenue from final_tbl 

By City

In [0]:
%sql

with final_tbl as (
    select  o.*,product_name,price,category,First_Name,Last_Name,Company,City,Country from orders o
left join(select product_id, product_name,price,category from products) p on o.product_id=p.product_id
left join (select Customer_Id,First_Name,Last_Name,Company,City,Country from customers) c on o.customer_id=c.customer_id
)


select city,product_id,quantity,price,sum( quantity*price) as total_revenue_product,sum(quantity*price) over(partition by city) as total_revenue_by_city from final_tbl group by city,product_id,quantity,price order by city



In [0]:
%sql

with final_tbl as (
    select  o.*,product_name,price,category,First_Name,Last_Name,Company,City,Country from orders o
left join(select product_id, product_name,price,category from products) p on o.product_id=p.product_id
left join (select Customer_Id,First_Name,Last_Name,Company,City,Country from customers) c on o.customer_id=c.customer_id
)

select city,sum( quantity*price) as total_revenue_by_city from final_tbl group by city order by city